# Advanced Models for Quora Question Pairs

This notebook demonstrates more advanced machine learning models and ensemble approaches.

## Setup and Data Loading

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score

# Add src to path
sys.path.insert(0, '..')

from src import (
    load_data,
    engineer_features,
    RandomForestModel,
    GradientBoostingModel,
    SVMModel,
    evaluate_model,
    print_evaluation_report
)

%matplotlib inline
sns.set_style('whitegrid')

### Load and Prepare Data

In [ ]:
# Load datasets
df_train, df_test = load_data(
    '../data/quora_question_pairs_train.csv.zip',
    '../data/quora_question_pairs_test.csv.zip'
)

# Engineer features
df_train = engineer_features(df_train)

# Prepare features
feature_cols = ['q1_len', 'q2_len', 'len_diff', 'common_words', 'jaccard_sim', 'word_match_share']
X = df_train[feature_cols]
y = df_train['is_duplicate']

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data prepared for training")
print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

## Random Forest Model

In [ ]:
# Train Random Forest
rf_model = RandomForestModel(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred = rf_model.predict(X_val)
y_pred_proba = rf_model.predict_proba(X_val)

metrics_rf = evaluate_model(y_val, y_pred, y_pred_proba)
print("\nRandom Forest Performance:")
print_evaluation_report(metrics_rf)

### Feature Importance (Random Forest)

In [ ]:
# Get feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importance()
}).sort_values('importance', ascending=False)

print(importance)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()

## Gradient Boosting Model

In [ ]:
# Train Gradient Boosting
gb_model = GradientBoostingModel(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred = gb_model.predict(X_val)
y_pred_proba = gb_model.predict_proba(X_val)

metrics_gb = evaluate_model(y_val, y_pred, y_pred_proba)
print("\nGradient Boosting Performance:")
print_evaluation_report(metrics_gb)

## Support Vector Machine

In [ ]:
# Note: SVM training can take time with large datasets
# Train SVM
svm_model = SVMModel(kernel='rbf', random_state=42)
print("Training SVM (this may take a while)...")
svm_model.fit(X_train, y_train)

# Predictions and evaluation
y_pred = svm_model.predict(X_val)
y_pred_proba = svm_model.predict_proba(X_val)

metrics_svm = evaluate_model(y_val, y_pred, y_pred_proba)
print("\nSVM Performance:")
print_evaluation_report(metrics_svm)

## Model Comparison

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Random Forest': [metrics_rf['accuracy'], metrics_rf['precision'], metrics_rf['recall'], metrics_rf['f1'], metrics_rf.get('roc_auc', 0)],
    'Gradient Boosting': [metrics_gb['accuracy'], metrics_gb['precision'], metrics_gb['recall'], metrics_gb['f1'], metrics_gb.get('roc_auc', 0)],
    'SVM': [metrics_svm['accuracy'], metrics_svm['precision'], metrics_svm['recall'], metrics_svm['f1'], metrics_svm.get('roc_auc', 0)],
}, index=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'])

print(comparison)

# Visualize comparison
comparison.plot(kind='bar', figsize=(12, 6))
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.legend()
plt.tight_layout()

## Save Best Model

In [ ]:
# Save the best performing model (usually Gradient Boosting)
gb_model.save('../models/best_model.pkl')
print("Best model saved to models/best_model.pkl")

## Summary

Comparing advanced models:
- **Random Forest**: Good interpretability, fast training, excellent feature importance
- **Gradient Boosting**: Often best performance, sequential training, better at capturing complex patterns
- **SVM**: Good with kernel tricks, but slower and memory-intensive on large datasets

Recommendations:
1. Use Gradient Boosting for best predictive performance
2. Use Random Forest for interpretability and faster training
3. Ensemble approaches combining multiple models can improve robustness